# 多項式回帰と過学習(演習)

解説資料は `research-handbook/machine-learning/ml-basics.md(および linear-regression.md)`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/polynomial_regression_solution.ipynb` にあるが、まず自力で取り組むこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

# PRML Ch.1 の例: sin(2πx) + ガウスノイズ
def make_data(n, noise=0.25, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    t = np.sin(2 * np.pi * x) + rng.normal(0, noise, n)
    return x, t

x_train, t_train = make_data(10, seed=1)
x_test,  t_test  = make_data(100, seed=2)   # 汎化誤差の推定用

## 課題1: 計画行列

多項式基底 $\phi_j(x) = x^j$ を並べた計画行列 $\Phi \in \mathbb{R}^{N \times (M+1)}$ を作ります。`np.vander` が便利です。

In [ ]:
def design_matrix(x, M):
    """課題1: 多項式基底の計画行列 Φ (N, M+1)。Φ[n, j] = x_n^j"""
    # ---- ここから課題1 ----
    # ヒント: np.vander(x, M+1, increasing=True)
    return None
    # ---- ここまで ----

Phi = design_matrix(np.array([0.0, 0.5, 1.0]), 2)
print(Phi)   # 各行が [1, x, x^2] になるはず

## 課題2: 最小二乗フィット

正規方程式 $\Phi^\top \Phi \, w = \Phi^\top t$ を解きます。**`np.linalg.inv` で逆行列を作らず `solve` を使う**こと(`linear-regression.md` 参照)。$M$ を変えると、訓練誤差は単調に下がるのにテスト誤差は $M=9$ で悪化する=過学習を観察します。

In [ ]:
def fit_least_squares(x, t, M):
    """課題2: 最小二乗解 w = (Φ^T Φ)^{-1} Φ^T t。
    逆行列は作らず solve を使うこと(数値安定性)"""
    Phi = design_matrix(x, M)
    # ---- ここから課題2 ----
    # 正規方程式 (Φ^T Φ) w = Φ^T t を np.linalg.solve で解く
    return None
    # ---- ここまで ----

def rmse(x, t, w, M):
    y = design_matrix(x, M) @ w
    return np.sqrt(np.mean((y - t) ** 2))

# 次数 M を変えて訓練誤差とテスト誤差を比較(過学習の観察)
for M in [0, 1, 3, 9]:
    w = fit_least_squares(x_train, t_train, M)
    print(f"M={M}: train RMSE={rmse(x_train, t_train, w, M):.3f}, "
          f"test RMSE={rmse(x_test, t_test, w, M):.3f}")

In [ ]:
xs = np.linspace(0, 1, 200)
plt.figure(figsize=(10, 3))
for i, M in enumerate([1, 3, 9]):
    w = fit_least_squares(x_train, t_train, M)
    plt.subplot(1, 3, i + 1)
    plt.scatter(x_train, t_train, c="b", s=20)
    plt.plot(xs, np.sin(2 * np.pi * xs), "g--", label="true")
    plt.plot(xs, design_matrix(xs, M) @ w, "r", label=f"M={M}")
    plt.ylim(-1.8, 1.8); plt.legend()
plt.tight_layout(); plt.show()

## 課題3: リッジ正則化

$w = (\lambda I + \Phi^\top \Phi)^{-1} \Phi^\top t$。$M=9$ のままでも $\lambda$ で複雑さを制御できること、$\lambda$ が大きすぎると今度は当てはまらなくなる(バイアス増)ことを確認します。

In [ ]:
def fit_ridge(x, t, M, lam):
    """課題3: リッジ回帰 w = (λI + Φ^T Φ)^{-1} Φ^T t
    (ガウス事前分布によるMAP推定と等価。probability-basics.md 参照)"""
    Phi = design_matrix(x, M)
    # ---- ここから課題3 ----
    # 課題2の正規方程式に λI を加えるだけ
    return None
    # ---- ここまで ----

# M=9(過学習する複雑さ)のまま λ で制御する
for lam in [0.0, 1e-6, 1e-3, 1.0]:
    w = fit_ridge(x_train, t_train, 9, lam) if lam > 0 else fit_least_squares(x_train, t_train, 9)
    print(f"lambda={lam:g}: test RMSE={rmse(x_test, t_test, w, 9):.3f}, |w|max={np.abs(w).max():.1f}")

## 発展: 交差検証で $\lambda$ を選ぶ

$\lambda$ の選択に**テストデータを使ってはいけません**(それはリーク。`model-evaluation.md`)。交差検証で選びます。

In [ ]:
# 発展: 5-fold交差検証で λ を選ぶ(テストデータは使わない!)
def cv_rmse(x, t, M, lam, k=5, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(x))
    folds = np.array_split(idx, k)
    errs = []
    for i in range(k):
        val = folds[i]
        trn = np.concatenate([folds[j] for j in range(k) if j != i])
        w = fit_ridge(x[trn], t[trn], M, lam)
        errs.append(rmse(x[val], t[val], w, M))
    return np.mean(errs)

x_big, t_big = make_data(50, seed=3)
lams = [1e-8, 1e-6, 1e-4, 1e-2, 1.0]
scores = [cv_rmse(x_big, t_big, 9, lam) for lam in lams]
best = lams[int(np.argmin(scores))]
print("CV RMSE:", dict(zip(lams, np.round(scores, 3))), "→ best lambda =", best)

## まとめ

- 過学習は「訓練誤差とテスト誤差の乖離」として観察できる
- 正則化はMAP推定(ガウス事前分布)と等価(`probability-basics.md`)
- ハイパーパラメータは交差検証で選び、テストは最後に1回だけ(`model-evaluation.md`)
- ベイズ線形回帰に進むと「予測の不確実性」まで得られる(`linear-regression.md` → `gaussian-process.md`)